# Sleep-EDF: frozen TinySleepNet voting118–123

Новый режим PhysioEx L20 усредняет logits всех позиций; прежний rolling сохранён. Обучение выключено.
Порядок: prepare smoke → prepare full → all (автоматически smoke, затем full). Модели используют двеT4, ансамбли — готовые вероятности.
Подключите full_psg_v1 и новый voting-inputs.zip. Прежние кэши106/embeddings128 не нужны.
Границы: все реальные полные окна; сегменты<20 получают явно помеченный native-short rolling fallback, без удаленияID.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_URL = "https://github.com/revlze/sleep-edf-classification.git"
CODE_REF = "feat/pretrained-refinement"
PROJECT_ROOT = Path("/tmp/sleep-edf-voting-code")
OUTPUT_ROOT = Path("/kaggle/working/voting-results")
CACHE_ROOT = Path("/tmp/sleep-edf-voting-cache")
INPUT_ROOTS = []  # Автопоиск нового sleep-edf-voting-inputs.zip / bundle.json.
CNN_INPUTS = []  # Необязательно: прежний voting-cache.zip / распакованный CNN-кэш.
RESUME_ROOT = None  # Распакованные результаты ЭТОЙ voting-серии.
RESUME_CACHE_ROOT = (
    None  # Необязательно: копировать прежний CNN-кэш в writable CACHE_ROOT.
)

RUN_GROUP = "prepare"  # prepare / models / ensembles / all / custom
CUSTOM_IDS = []
SMOKE_ONLY = True
SMOKE_EPOCHS = 160
DEVICE = "cuda:0"
THREADS = min(4, os.cpu_count() or 2)
PARALLEL_DEVICES = [
    "cuda:0",
    "cuda:1",
]  # Две независимые очереди моделей; [] для однойGPU.
EXPORT_CNN_CACHE = (
    True  # Отдельный накопительный ZIP; ~1.8GB до сжатия полногоCNN-кэша.
)

GROUPS = {
    "prepare": [],
    "models": [118, 119, 120],
    "ensembles": [121, 122, 123],
    "all": list(range(118, 124)),
}
CONFIG_IDS = (
    CUSTOM_IDS
    if RUN_GROUP == "custom"
    else [f"{n:03d}_pretrained" for n in GROUPS[RUN_GROUP]]
)
if RUN_GROUP != "prepare" and (
    not CONFIG_IDS
    or any(
        n not in [f"{i:03d}_pretrained" for i in range(118, 124)] for n in CONFIG_IDS
    )
):
    raise ValueError("Нужны полные ID118_pretrained …123_pretrained")


def checkout_code():
    if PROJECT_ROOT.exists():
        current_ref = subprocess.check_output(
            ["git", "config", "sleepedf.bootstrapRef"], cwd=PROJECT_ROOT, text=True
        ).strip()
        if current_ref != CODE_REF:
            raise RuntimeError(
                "Для другого CODE_REF задайте новый PROJECT_ROOT в /tmp."
            )
        print(
            "Используется существующий checkout; для обновления ветки задайте новый PROJECT_ROOT."
        )
        return
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    with tempfile.TemporaryDirectory(prefix="sleep-git-") as temporary:
        askpass = Path(temporary) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\nimport os, sys\n"
            'print("x-access-token" if "Username" in sys.argv[1] else os.environ["SLEEP_GITHUB_TOKEN"])\n'
        )
        askpass.chmod(0o700)
        env = {
            **os.environ,
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "SLEEP_GITHUB_TOKEN": token,
        }
        subprocess.run(
            ["git", "init", "--quiet", "--initial-branch=bootstrap", str(PROJECT_ROOT)],
            check=True,
        )
        subprocess.run(
            ["git", "remote", "add", "origin", REPO_URL], cwd=PROJECT_ROOT, check=True
        )
        subprocess.run(
            [
                "git",
                "-c",
                "credential.helper=",
                "fetch",
                "--quiet",
                "--depth",
                "1",
                "origin",
                CODE_REF,
            ],
            cwd=PROJECT_ROOT,
            env=env,
            check=True,
        )
        subprocess.run(
            ["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"],
            cwd=PROJECT_ROOT,
            check=True,
        )
        subprocess.run(
            ["git", "config", "sleepedf.bootstrapRef", CODE_REF],
            cwd=PROJECT_ROOT,
            check=True,
        )
    del token


checkout_code()
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
print("Код:", commit)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook import reset_commands, run_command  # noqa: E402

reset_commands()

# Устанавливаем только инструмент uv; зависимости проекта устанавливает uv.lock.

UV_VERSION = "0.12.15"
if shutil.which("uv") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", f"uv=={UV_VERSION}"], check=True
    )
UV = shutil.which("uv")
ENV = {
    "UV_PROJECT_ENVIRONMENT": "/tmp/sleep-edf-refinement-venv",
    "UV_CACHE_DIR": "/tmp/sleep-pretrained-uv-cache",
    "UV_PYTHON_INSTALL_DIR": "/tmp/sleep-pretrained-python",
    "PYTHONPATH": str(PROJECT_ROOT),
    "MPLBACKEND": "Agg",  # Backend только для дочернего CLI, не Kaggle-ядра.
}
UV_PROJECT = PROJECT_ROOT / "environments/pretrained"
print("Ядро notebook:", sys.version)
subprocess.run(["nvidia-smi"], check=True)
run_command(
    [
        UV,
        "sync",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "--python",
        "3.12.12",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)
BASE = [
    UV,
    "run",
    "--project",
    str(UV_PROJECT),
    "--locked",
    "--no-dev",
    "python",
    "-m",
    "src.voting_run",
]
ENV.update(
    {
        k: str(THREADS)
        for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")
    }
)
run_command(
    [
        UV,
        "run",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "python",
        "-c",
        "from src.pretrained import hardware; import json; print(json.dumps(hardware("
        + repr(DEVICE)
        + "),indent=2))",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)
print(
    "Группа:", RUN_GROUP, " · обучение выключено · CNN batch256, LSTM window batch512"
)

In [ ]:
import hashlib
from zipfile import ZipFile

import pandas as pd
from IPython.display import display


def unpack(archive):
    key = hashlib.sha256(
        (str(archive) + str(archive.stat().st_size)).encode()
    ).hexdigest()[:16]
    target = Path("/tmp/sleep-edf-voting-inputs") / key
    if (target / ".unpacked").exists():
        return target
    target.mkdir(parents=True, exist_ok=True)
    with ZipFile(archive) as zipped:
        for info in zipped.infolist():
            destination = (target / info.filename).resolve()
            if (
                not destination.is_relative_to(target.resolve())
                or (info.external_attr >> 16) & 0o170000 == 0o120000
            ):
                raise ValueError("Небезопасный ZIP path")
            zipped.extract(info, target)
    (target / ".unpacked").touch()
    return target


roots = [
    p.parent
    for p in Path("/kaggle/input").rglob("MANIFEST.json")
    if (p.parent / "labels.csv").is_file()
    and (p.parent / "epochs/val").is_dir()
    and (p.parent / "epochs/test").is_dir()
]
if len(roots) != 1:
    raise RuntimeError(f"Нужен один исходный full_psg_v1 dataset, найдено{roots}")
DATA_ROOT = roots[0]
need_inputs, need_cnn = not INPUT_ROOTS, not CNN_INPUTS
if need_inputs or need_cnn:
    for filename, kind, destination, needed in [
        ("bundle.json", "tiny_voting_inputs_v1", INPUT_ROOTS, need_inputs),
        ("cache_manifest.json", "tiny_voting_cnn_v1", CNN_INPUTS, need_cnn),
    ]:
        if not needed:
            continue
        candidates = []
        for p in Path("/kaggle/input").rglob(filename):
            if json.loads(p.read_text()).get("kind") == kind:
                candidates.append(p.parent)
        if not candidates:
            for archive in Path("/kaggle/input").rglob("*.zip"):
                with ZipFile(archive) as zipped:
                    if (
                        filename in zipped.namelist()
                        and json.loads(zipped.read(filename)).get("kind") == kind
                    ):
                        candidates.append(unpack(archive))
        destination.extend(sorted(set(candidates)))
if not INPUT_ROOTS:
    raise FileNotFoundError(
        "Подключите новый sleep-edf-voting-inputs.zip: в нём уже все нужные веса и старые вероятности"
    )
for previous, target in [(RESUME_ROOT, OUTPUT_ROOT), (RESUME_CACHE_ROOT, CACHE_ROOT)]:
    if previous is not None:
        if target.exists():
            print("Существующий каталог сохранён:", target)
        else:
            shutil.copytree(Path(previous), target)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
COMMON = [
    "--data-root",
    str(DATA_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
    "--cache-root",
    str(CACHE_ROOT),
    "--device",
    DEVICE,
    "--threads",
    str(THREADS),
]
for p in INPUT_ROOTS:
    COMMON += ["--input-root", str(p)]
for p in CNN_INPUTS:
    COMMON += ["--cnn-input", str(p)]
for name in CONFIG_IDS:
    if not (PROJECT_ROOT / "configs" / (name + ".yaml")).is_file():
        raise FileNotFoundError(
            name + ": сначала нужен опубликованный актуальныйCODE_REF"
        )
display(
    pd.DataFrame(
        [
            {
                "ID": n,
                "status": "complete"
                if (
                    OUTPUT_ROOT / ("smoke" if SMOKE_ONLY else "") / n / "metrics.json"
                ).exists()
                else "pending/partial",
            }
            for n in CONFIG_IDS
        ]
    )
)
print(
    "DATA_ROOT:", DATA_ROOT, "\nINPUT_ROOTS:", INPUT_ROOTS, "\nCNN_INPUTS:", CNN_INPUTS
)
print(
    "prepare — только два frozen CNN backbone по val/test; обучение, CV и подбор весов отсутствуют."
)
print(
    "Сегменты<20: сохраняется native rolling fallback, official vote_count0; отдельные covered-only метрики. Всеepoch ID сохранены."
)

In [ ]:
def run_one(names, smoke, prepare=False, device=None, tag=None):
    import time

    start = time.time_ns()
    command = [
        *BASE,
        *[str(PROJECT_ROOT / "configs" / (n + ".yaml")) for n in names],
        *COMMON,
    ]
    env = ENV.copy()
    if device is not None:
        command[command.index("--device") + 1] = device
        threads = max(1, min(THREADS, (os.cpu_count() or 2) // len(PARALLEL_DEVICES)))
        command[command.index("--threads") + 1] = str(threads)
        env.update(
            {
                k: str(threads)
                for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")
            }
        )
    if tag:
        command += ["--report-tag", tag]
    if smoke:
        command += ["--smoke", "--smoke-epochs", str(SMOKE_EPOCHS)]
    if prepare:
        command += ["--prepare"]
    try:
        run_command(command, cwd=PROJECT_ROOT, env=env)
    except subprocess.CalledProcessError:
        print(
            "Есть ошибки. Завершённые записи сохранены; детали в<ID>/last_error.json."
        )
        if prepare:
            raise
    namespace = OUTPUT_ROOT / "smoke" if smoke else OUTPUT_ROOT
    report = namespace / (
        f"voting_failures-{tag}.json" if tag else "voting_failures.json"
    )
    fresh = report.exists() and report.stat().st_mtime_ns >= start
    failed = (
        {r["experiment_id"] for r in json.loads(report.read_text())}
        if fresh
        else set(names)
    )
    return [
        n
        for n in names
        if n not in failed and (namespace / n / "metrics.json").exists()
    ]


def run_batch(names, smoke, prepare=False):
    from src.notebook import cancel_commands, reset_commands

    reset_commands()
    models = [n for n in names if 118 <= int(n[:3]) <= 120]
    ensembles = [n for n in names if 121 <= int(n[:3]) <= 123]
    if prepare:
        return run_one([], smoke, True)
    passed = []
    if models and len(PARALLEL_DEVICES) >= 2:
        namespace = OUTPUT_ROOT / "smoke" if smoke else OUTPUT_ROOT
        if not (namespace / "prepare_report.json").exists():
            raise RuntimeError(
                "Для двух GPU сначала выполните prepare в этом режиме; либо PARALLEL_DEVICES=[]"
            )
        if len(PARALLEL_DEVICES) != 2 or len(set(PARALLEL_DEVICES)) != 2:
            raise ValueError("Нужны два разных устройства или PARALLEL_DEVICES=[]")
        from concurrent.futures import ThreadPoolExecutor

        pool = ThreadPoolExecutor(max_workers=2)
        try:
            futures = [
                pool.submit(run_one, models[i::2], smoke, False, device, f"gpu{i}")
                for i, device in enumerate(PARALLEL_DEVICES)
                if models[i::2]
            ]
            passed.extend(n for future in futures for n in future.result())
        except BaseException:
            cancel_commands()
            raise
        finally:
            pool.shutdown(wait=True, cancel_futures=True)
    elif models:
        passed.extend(run_one(models, smoke))
    # Explicit barrier: fixed mixtures start only after both GPU queues finish.
    if ensembles:
        passed.extend(run_one(ensembles, smoke))
    return [n for n in names if n in passed]


if RUN_GROUP == "prepare":
    run_batch([], True, prepare=True)
    if not SMOKE_ONLY:
        run_batch([], False, prepare=True)
else:
    passed = run_batch(CONFIG_IDS, True)
    if not SMOKE_ONLY:
        run_batch(passed, False)
namespace = OUTPUT_ROOT / "smoke" if SMOKE_ONLY else OUTPUT_ROOT
rows = []
for name in CONFIG_IDS:
    path = namespace / name / "metrics.json"
    if path.exists():
        m = json.loads(path.read_text())
        rows.append(
            {
                "ID": name,
                "mode": "SMOKE" if m["smoke"] else "VAL+TEST",
                "val_macro_f1": m["validation"]["macro_f1"],
                "inference_s": m["inference_seconds"],
                "GPU_peak_GiB": m["peak_gpu_bytes"] / 2**30
                if m["peak_gpu_bytes"]
                else None,
            }
        )
    else:
        error = namespace / name / "last_error.json"
        rows.append(
            {
                "ID": name,
                "mode": "FAILED/PARTIAL",
                "error": json.loads(error.read_text())["error"]
                if error.exists()
                else "not completed",
            }
        )
display(pd.DataFrame(rows))

После остановки можно выполнить экспорт последней ячейкой. ZIP результатов и CNN-кэша накопительные; веса не обучаются и повторно не экспортируются.
Для новой сессии распакуйте результаты и задайтеRESUME_ROOT. Voting CNN-кэш обнаруживается автоматически вInput и читается без копирования. Сохраняйте ту же версию кода для resume.


In [ ]:
from datetime import UTC, datetime

from IPython.display import FileLink, display

from src.notebook import reset_commands

# Stop cancels both GPU queues; permit this independent export afterwards.
reset_commands()

stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
archive = Path("/kaggle/working") / f"sleep-edf-voting-results-{stamp}.zip"
run_command(
    [*BASE, "--output-root", str(OUTPUT_ROOT), "--export", str(archive)],
    cwd=PROJECT_ROOT,
    env=ENV,
)
display(
    FileLink(
        str(archive.relative_to(Path.cwd())),
        result_html_prefix="Накопительные результаты, сравнения и submissions: ",
    )
)
if EXPORT_CNN_CACHE and (CACHE_ROOT.exists() or CNN_INPUTS):
    target = Path("/kaggle/working") / f"sleep-edf-voting-cache-{stamp}.zip"
    command = [*BASE, "--cache-root", str(CACHE_ROOT), "--export-cache", str(target)]
    for p in CNN_INPUTS:
        command += ["--cnn-input", str(p)]
    run_command(command, cwd=PROJECT_ROOT, env=ENV)
    display(
        FileLink(
            str(target.relative_to(Path.cwd())),
            result_html_prefix="Накопительный CNN-кэш для следующих сессий: ",
        )
    )